<a href="https://colab.research.google.com/github/kalyandrug/rk/blob/main/clinical_data_diabetic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

RANDOM_STATE = 42

In [ ]:
#upload file
from google.colab import files

# Upload the file auto csv file
uploaded = files.upload()

# Get the filename
filename = list(uploaded.keys())[0]
print(f"Uploaded file: {filename}")

In [ ]:
# -----------------------------------------------------
# 1. Load data
# -----------------------------------------------------
# Option A: use your own auto MPG CSV (uncomment and edit path)
df = pd.read_csv(filename, na_values='?', comment='\t', sep=',', skipinitialspace=True)

In [ ]:
#display datasets
print(df)

In [ ]:
df.describe()

In [ ]:
# 1. Identify missing values in the dataset.
print("Missing values in the dataset:\n", df.isnull().sum())

In [ ]:
# 3. REMOVE EXACT DUPLICATE ROWS
# ---------------------------------------------------------------------------
dupes_before = df.duplicated().sum()
df = df.drop_duplicates()
print(f"\nDropped {dupes_before} exact duplicate rows. New shape: {df.shape}")

In [ ]:
# 4. CONVERT DISGUISED ZEROS TO NaN
#    These columns cannot legitimately be 0 in a living patient.
#    'pregnancies' and 'outcome' are excluded — 0 pregnancies and 0 outcome
#    (non-diabetic) are valid real values.
# ---------------------------------------------------------------------------
zero_as_missing_cols = ["glucose", "blood_pressure", "skin_thickness", "insulin", "bmi"]

print("\nZero-value counts per column BEFORE conversion (likely disguised missing data):")
print((df[zero_as_missing_cols] == 0).sum())

for col in zero_as_missing_cols:
    df[col] = df[col].replace(0, np.nan)

print("\nMissing value counts AFTER converting disguised zeros:")
print(df[zero_as_missing_cols].isna().sum())
print(f"\n'insulin' missing rate: {df['insulin'].isna().mean():.1%}")
print(f"'skin_thickness' missing rate: {df['skin_thickness'].isna().mean():.1%}")

In [ ]:
# ---------------------------------------------------------------------------
# 5. OUTLIER / RANGE CHECK
#    Flag biologically implausible values that survived (too high, not just
#    too low). This dataset is fairly clean, but always verify.
# ---------------------------------------------------------------------------
range_checks = {
    "glucose": (40, 300),          # mg/dL
    "blood_pressure": (30, 180),   # mmHg (diastolic)
    "bmi": (10, 70),
    "age": (18, 100),
}
print("\nOut-of-range value counts (values outside plausible clinical range):")
for col, (low, high) in range_checks.items():
    out_of_range = ((df[col] < low) | (df[col] > high)).sum()
    print(f"  {col}: {out_of_range} rows outside [{low}, {high}]")

In [ ]:

# ---------------------------------------------------------------------------
# 6. SAVE CLEANED (BUT NOT YET IMPUTED) DATASET
#    We deliberately do NOT impute here. Imputation should be fit on the
#    training split only, inside your modeling pipeline — not on the full
#    dataset — to avoid data leakage. This script hands off a clean,
#    correctly-typed, duplicate-free, properly-NaN-flagged CSV.
# ---------------------------------------------------------------------------
df.to_csv("diabetes_clinical_cleaned.csv", index=False)
print("\nSaved cleaned dataset to diabetes_clinical_cleaned.csv")
print("Final shape:", df.shape)
print("\nRemaining missing values (to be imputed inside train/test split during modeling):")
print(df.isna().sum())

In [ ]:
"""
Diabetes Prediction — End-to-End ML Pipeline
===============================================
Dataset: diabetes_clinical_cleaned.csv (768 patients, Pima Indians Diabetes)
Target : outcome (1 = diabetic, 0 = not diabetic)

This script assumes diabetes_data_cleanup.py has already been run — the
input CSV has disguised zeros already converted to NaN, and duplicates
already removed. This script handles imputation INSIDE the pipeline
(fit on train split only) to avoid data leakage, then trains and compares
Logistic Regression vs Random Forest.
"""

# ---------------------------------------------------------------------------
# 2. DEFINE FEATURES / TARGET
# ---------------------------------------------------------------------------
TARGET = "outcome"
feature_cols = [c for c in df.columns if c != TARGET]

X = df[feature_cols]
y = df[TARGET]

In [ ]:

# ---------------------------------------------------------------------------
# 3. TRAIN/TEST SPLIT (stratified — preserves ~35% diabetic ratio in both sets)
# ---------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"\nTrain size: {X_train.shape[0]}  Test size: {X_test.shape[0]}")


In [ ]:

# ---------------------------------------------------------------------------
# 4. PIPELINES
#    All features here are numeric, so preprocessing is simpler than the
#    churn pipeline (no ColumnTransformer needed for mixed types).
#    - Logistic Regression: impute -> scale -> classify
#    - Random Forest: impute -> classify (tree models don't need scaling)
# ---------------------------------------------------------------------------
log_reg_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

rf_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=300, max_depth=6,
        class_weight="balanced", random_state=RANDOM_STATE
    )),
])

In [ ]:
# ---------------------------------------------------------------------------
# 5. TRAIN
# ---------------------------------------------------------------------------
log_reg_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)


In [ ]:
# ---------------------------------------------------------------------------
# 6. CROSS-VALIDATION (5-fold, on training data only)
#    Important on a small dataset like this (768 rows) — a single train/test
#    split can be misleading, so CV gives a more reliable performance estimate.
# ---------------------------------------------------------------------------
cv_lr = cross_val_score(log_reg_pipeline, X_train, y_train, cv=5, scoring="roc_auc")
cv_rf = cross_val_score(rf_pipeline, X_train, y_train, cv=5, scoring="roc_auc")

print("\n5-fold CV AUROC — Logistic Regression:", np.round(cv_lr, 3), "mean:", round(cv_lr.mean(), 3))
print("5-fold CV AUROC — Random Forest:      ", np.round(cv_rf, 3), "mean:", round(cv_rf.mean(), 3))

In [ ]:

# ---------------------------------------------------------------------------
# 7. EVALUATE ON HELD-OUT TEST SET
# ---------------------------------------------------------------------------
def evaluate(name, pipeline, X_test, y_test):
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    print(f"\n=== {name} — Test Set Performance ===")
    print(f"Accuracy:    {accuracy_score(y_test, y_pred):.3f}")
    print(f"AUROC:       {roc_auc_score(y_test, y_proba):.3f}")
    print(f"Sensitivity (Recall, catches diabetics): {recall_score(y_test, y_pred):.3f}")
    print(f"Precision:   {precision_score(y_test, y_pred):.3f}")
    print(f"F1:          {f1_score(y_test, y_pred):.3f}")
    print("Confusion matrix [[TN FP][FN TP]]:")
    print(confusion_matrix(y_test, y_pred))
    return y_pred, y_proba

evaluate("Logistic Regression", log_reg_pipeline, X_test, y_test)
evaluate("Random Forest", rf_pipeline, X_test, y_test)


In [ ]:

# ---------------------------------------------------------------------------
# 8. INTERPRETABILITY — coefficients (LR) and feature importances (RF)
# ---------------------------------------------------------------------------
lr_coefs = log_reg_pipeline.named_steps["classifier"].coef_[0]
coef_df = pd.DataFrame({"feature": feature_cols, "coef": lr_coefs}) \
    .sort_values("coef", key=abs, ascending=False)

print("\n=== Logistic Regression coefficients (standardized) ===")
print("Positive = increases diabetes risk, Negative = decreases risk")
print(coef_df.to_string(index=False))

rf_importances = rf_pipeline.named_steps["classifier"].feature_importances_
imp_df = pd.DataFrame({"feature": feature_cols, "importance": rf_importances}) \
    .sort_values("importance", ascending=False)

print("\n=== Random Forest feature importances ===")
print(imp_df.to_string(index=False))

In [ ]:

# ---------------------------------------------------------------------------
# 9. PREDICT ON NEW / UNSEEN PATIENTS (example usage)
# ---------------------------------------------------------------------------
new_patients = X_test.head(3)
rf_preds = rf_pipeline.predict(new_patients)
rf_probs = rf_pipeline.predict_proba(new_patients)[:, 1]

print("\n=== Example: scoring new patients with Random Forest ===")
for i, (pred, prob) in enumerate(zip(rf_preds, rf_probs)):
    label = "Diabetic risk" if pred == 1 else "Low risk"
    print(f"Patient {i+1}: prediction={pred} ({label}), probability={prob:.2%}")

In [ ]:
"""
Diabetes Prediction — Graph Generation
=========================================
Loads the trained pipelines from diabetes_model.py logic and produces:
  1. ROC curves (Logistic Regression vs Random Forest)
  2. Confusion matrix heatmaps (side by side)
  3. Feature importance / coefficient comparison
  4. Missing data bar chart (from the cleanup step)

Saves all charts as PNG files.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")

# ---------------------------------------------------------------------------
# 1. LOAD + SPLIT (same as diabetes_model.py)
# ---------------------------------------------------------------------------
df = pd.read_csv("diabetes_clinical_cleaned.csv")
TARGET = "outcome"
feature_cols = [c for c in df.columns if c != TARGET]
X = df[feature_cols]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# ---------------------------------------------------------------------------
# 2. REBUILD + TRAIN PIPELINES
# ---------------------------------------------------------------------------
log_reg_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
rf_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=300, max_depth=6,
        class_weight="balanced", random_state=RANDOM_STATE
    )),
])
log_reg_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

y_proba_lr = log_reg_pipeline.predict_proba(X_test)[:, 1]
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]
y_pred_lr = log_reg_pipeline.predict(X_test)
y_pred_rf = rf_pipeline.predict(X_test)

# ---------------------------------------------------------------------------
# CHART 1: ROC CURVES
# ---------------------------------------------------------------------------
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
auc_lr = roc_auc_score(y_test, y_proba_lr)
auc_rf = roc_auc_score(y_test, y_proba_rf)

plt.figure(figsize=(7, 6))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {auc_lr:.3f})", linewidth=2, color="#2563eb")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {auc_rf:.3f})", linewidth=2, color="#dc2626")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC Curve — Diabetes Prediction Models")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=150)
plt.close()
print("Saved roc_curve.png")

# ---------------------------------------------------------------------------
# CHART 2: CONFUSION MATRICES SIDE BY SIDE
# ---------------------------------------------------------------------------
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_rf = confusion_matrix(y_test, y_pred_rf)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels = ["No Diabetes", "Diabetes"]

sns.heatmap(cm_lr, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=axes[0], cbar=False)
axes[0].set_title("Logistic Regression")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Reds", xticklabels=labels, yticklabels=labels, ax=axes[1], cbar=False)
axes[1].set_title("Random Forest")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.suptitle("Confusion Matrices — Test Set (n=154)")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.close()
print("Saved confusion_matrices.png")

# ---------------------------------------------------------------------------
# CHART 3: FEATURE IMPORTANCE COMPARISON
# ---------------------------------------------------------------------------
lr_coefs = log_reg_pipeline.named_steps["classifier"].coef_[0]
rf_importances = rf_pipeline.named_steps["classifier"].feature_importances_

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

coef_df = pd.DataFrame({"feature": feature_cols, "coef": lr_coefs}).sort_values("coef")
colors = ["#dc2626" if c < 0 else "#2563eb" for c in coef_df["coef"]]
axes[0].barh(coef_df["feature"], coef_df["coef"], color=colors)
axes[0].set_title("Logistic Regression Coefficients\n(blue = increases risk, red = decreases risk)")
axes[0].set_xlabel("Standardized coefficient")
axes[0].axvline(0, color="black", linewidth=0.8)

imp_df = pd.DataFrame({"feature": feature_cols, "importance": rf_importances}).sort_values("importance")
axes[1].barh(imp_df["feature"], imp_df["importance"], color="#16a34a")
axes[1].set_title("Random Forest Feature Importances")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
#plt.close()
print("Saved feature_importance.png")

# ---------------------------------------------------------------------------
# CHART 4: MISSING DATA OVERVIEW
# ---------------------------------------------------------------------------
missing_pct = (df[feature_cols].isna().sum() / len(df) * 100).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
bars = plt.bar(missing_pct.index, missing_pct.values, color="#f59e0b")
plt.ylabel("% Missing")
plt.title("Missing Data by Feature (after fixing disguised zeros)")
plt.xticks(rotation=45, ha="right")
for bar, pct in zip(bars, missing_pct.values):
    if pct > 0:
        plt.text(bar.get_x() + bar.get_width()/2, pct + 0.5, f"{pct:.1f}%", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("missing_data.png", dpi=150)
plt.close()
print("Saved missing_data.png")

print("\nAll charts generated successfully.")